# Partial Dependence Analysis: Heat Stress Mechanisms and Cattle Mortality

## Scientific Background

**Partial Dependence Plots (PDPs)** reveal the marginal effect of specific features on model predictions while averaging over all other features. This technique is essential for:

1. **Understanding non-linear relationships**: How mortality risk changes across the range of heat metrics
2. **Identifying critical thresholds**: Points where small changes in conditions lead to large mortality increases
3. **Quantifying interaction effects**: How multiple stressors combine synergistically
4. **Validating biological mechanisms**: Confirming theoretical heat stress pathways
5. **Informing management decisions**: Identifying actionable intervention points

### Key Heat Stress Variables

**Daytime Heat Exposure:**
- Hours above 30°C (moderate heat stress threshold)
- Hours above 35°C (severe heat stress threshold)
- Direct metabolic heat load and thermoregulatory burden

**Nighttime Recovery:**
- Hours above 21°C (poor recovery threshold)
- Hours above 24°C (very poor recovery threshold)
- Critical for heat dissipation and physiological restoration

**Vapor Pressure Deficit (VPD):**
- Mean VPD (kPa) - sustained evaporative stress
- Max VPD (kPa) - peak evaporative demand
- Impairs evaporative cooling (panting, sweating)

### Biological Hypotheses

**H1 (Threshold Effects):** Mortality risk increases non-linearly, with critical thresholds at:
- Daytime: 20-25 hours/week above 30°C
- Nighttime: 15-20 hours/week above 21°C
- VPD: 2.0-2.5 kPa mean

**H2 (Synergistic Interactions):** Combined heat + VPD effects exceed additive predictions

**H3 (Recovery Dependency):** Nighttime recovery moderates daytime heat impacts

**H4 (Asymmetric Responses):** Mortality increases faster above thresholds than it decreases below them

### Analysis Plan

1. **1D Partial Dependence Plots**: Individual feature effects
2. **2D Partial Dependence Plots**: Pairwise interaction effects
3. **Individual Conditional Expectation (ICE) Plots**: Prediction variability across samples
4. **Threshold Detection**: Statistical identification of critical points
5. **Uncertainty Quantification**: Confidence intervals via bootstrapping
6. **Biological Validation**: Compare with livestock physiology literature

---

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Machine learning libraries
from sklearn.inspection import partial_dependence, PartialDependenceDisplay
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import pickle

# Import project configuration
import sys
sys.path.append('../../')
from config import (
    STATE_NAMES, STATE_ABBRS, STATE_REGIONS, 
    FOCUS_STATES, CATTLE_REGIONS, CUSTOM_REGIONS, SEASONS
)

# Set plotting style
sns.set_style('whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries loaded successfully")

Libraries loaded successfully


## 1. Load Trained Model and Data

In [2]:
from pathlib import Path

# Define paths
CATTLE_DATA_DIR = Path('../../cattle_data')
FIGURES_DIR = Path('../../figures')
OUTPUT_DIR = FIGURES_DIR / 'partial_dependence'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading trained model from 14_predictive_modeling_early_warning.ipynb...")

# Try to find the best model file (check what was actually saved)
possible_models = [
    ('Ridge Regression', 'best_model_ridge_regression.pkl'),
    ('Random Forest', 'best_model_random_forest.pkl'),
    ('Gradient Boosting', 'best_model_gradient_boosting.pkl'),
    ('Linear Regression', 'best_model_linear_regression.pkl')
]

model = None
model_name = None
model_file = None

for name, filename in possible_models:
    test_file = CATTLE_DATA_DIR / filename
    if test_file.exists():
        print(f"Found model: {name} ({filename})")
        model_file = test_file
        model_name = name
        with open(model_file, 'rb') as f:
            model = pickle.load(f)
        break

if model is None:
    raise FileNotFoundError(
        f"No trained model found in {CATTLE_DATA_DIR}\n"
        "Please run 14_predictive_modeling_early_warning.ipynb first to train a model.\n"
        f"Looked for: {[f[1] for f in possible_models]}"
    )

print(f"\nModel loaded: {model_name}")
print(f"  Type: {type(model).__name__}")

# Check if model supports partial dependence (tree-based models work best)
if hasattr(model, 'feature_importances_'):
    print(f"  ✓ Tree-based model - excellent for partial dependence analysis")
    print(f"  n_estimators: {getattr(model, 'n_estimators', 'N/A')}")
    print(f"  max_depth: {getattr(model, 'max_depth', 'N/A')}")
elif hasattr(model, 'coef_'):
    print(f"  ⚠ Linear model - partial dependence plots will show linear relationships")
    print(f"  Note: PD plots are most informative for tree-based models (Random Forest, Gradient Boosting)")
else:
    print(f"  Model type: {type(model).__name__}")

# Load scaler
scaler_file = CATTLE_DATA_DIR / 'feature_scaler.pkl'
if not scaler_file.exists():
    raise FileNotFoundError(f"Scaler file not found: {scaler_file}")

with open(scaler_file, 'rb') as f:
    scaler = pickle.load(f)
print("\nScaler loaded")

# Load feature list
feature_list_file = CATTLE_DATA_DIR / 'model_features.csv'
if not feature_list_file.exists():
    raise FileNotFoundError(f"Feature list not found: {feature_list_file}")

feature_list_df = pd.read_csv(feature_list_file)
all_features = feature_list_df['Feature'].tolist()
print(f"Total features: {len(all_features)}")

# Load predictions for analysis
predictions_file = CATTLE_DATA_DIR / 'model_predictions.csv'
if not predictions_file.exists():
    raise FileNotFoundError(f"Predictions file not found: {predictions_file}")

predictions_df = pd.read_csv(predictions_file, index_col=0)
print(f"Predictions loaded: {len(predictions_df)} samples")

print("\n" + "="*80)
print("NOTE: Partial Dependence Analysis")
print("="*80)
if hasattr(model, 'coef_'):
    print("The loaded model is linear (Ridge/Lasso/Linear Regression).")
    print("Partial dependence plots will show linear relationships.")
    print("For more interesting non-linear patterns, consider retraining with Random Forest.")
else:
    print("The loaded model is tree-based - ideal for partial dependence analysis!")
print("="*80)

Loading trained model from 14_predictive_modeling_early_warning.ipynb...
Found model: Ridge Regression (best_model_ridge_regression.pkl)

Model loaded: Ridge Regression
  Type: Ridge
  ⚠ Linear model - partial dependence plots will show linear relationships
  Note: PD plots are most informative for tree-based models (Random Forest, Gradient Boosting)

Scaler loaded
Total features: 51
Predictions loaded: 1044 samples

NOTE: Partial Dependence Analysis
The loaded model is linear (Ridge/Lasso/Linear Regression).
Partial dependence plots will show linear relationships.
For more interesting non-linear patterns, consider retraining with Random Forest.


In [ ]:
# Check if we need to train a Random Forest for partial dependence analysis
train_rf_for_pdp = not hasattr(model, 'feature_importances_')

if train_rf_for_pdp:
    print("\n" + "="*80)
    print("TRAINING RANDOM FOREST FOR PARTIAL DEPENDENCE ANALYSIS")
    print("="*80)
    print(f"Current model ({model_name}) is linear and will show linear relationships.")
    print("Training a Random Forest to capture non-linear effects and interactions...")
    print("")
    
    # Import sklearn modules
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
    
    # We'll load and prepare the full dataset in the next cells
    print("✓ Will train Random Forest after loading data")
    print("="*80)
else:
    print("\n" + "="*80)
    print(f"Model is tree-based ({model_name}) - excellent for partial dependence!")
    print("No need to train a new model.")
    print("="*80)

## 1.5. Train Random Forest Model for Partial Dependence Analysis (if needed)

If the loaded model is linear (Ridge/Lasso/Linear Regression), we'll train a Random Forest specifically for this analysis to capture non-linear relationships and interactions.

In [ ]:
# Load the full dataset used for modeling
print("Loading full dataset...")

# We need to reconstruct the dataset from the cattle-heat merged data
import xarray as xr
from config import CATTLE_REGION_IDS, PROCESSED_WEEKLY_DIR, MASK_FILE, CATTLE_DATA_FILE

# Load climate data
ds_night = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_nighttime_recovery.nc')
ds_day = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_daytime_heat.nc')
ds_vpd = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_vpd.nc')

week_dates = ds_night['week'].values

# Load state mask
ds_mask = xr.open_dataset(MASK_FILE)
state_mask = ds_mask.state_mask.load()
ds_mask.close()

# Helper function
def compute_regional_mean(data, state_ids, state_mask):
    combined_mask = xr.zeros_like(state_mask, dtype=bool)
    for state_id in state_ids:
        combined_mask = combined_mask | (state_mask == state_id)
    masked_data = data.where(combined_mask)
    return masked_data.mean(dim=['lat', 'lon']).astype(np.float64)

region_4_ids = CATTLE_REGION_IDS['region_4']
region_6_ids = CATTLE_REGION_IDS['region_6']

# Compute regional metrics
r4_metrics = pd.DataFrame({
    'week_ending': pd.to_datetime(week_dates),
    'region': 4,
    'mean_daytime_hours_above_30': compute_regional_mean(ds_day['hours_above_30'], region_4_ids, state_mask).values,
    'mean_daytime_hours_above_35': compute_regional_mean(ds_day['hours_above_35'], region_4_ids, state_mask).values,
    'mean_nighttime_hours_above_21': compute_regional_mean(ds_night['hours_above_21'], region_4_ids, state_mask).values,
    'mean_nighttime_hours_above_24': compute_regional_mean(ds_night['hours_above_24'], region_4_ids, state_mask).values,
    'mean_vpd_mean': compute_regional_mean(ds_vpd['vpd_mean'], region_4_ids, state_mask).values,
    'mean_vpd_max': compute_regional_mean(ds_vpd['vpd_max'], region_6_ids, state_mask).values,
})

r6_metrics = pd.DataFrame({
    'week_ending': pd.to_datetime(week_dates),
    'region': 6,
    'mean_daytime_hours_above_30': compute_regional_mean(ds_day['hours_above_30'], region_6_ids, state_mask).values,
    'mean_daytime_hours_above_35': compute_regional_mean(ds_day['hours_above_35'], region_6_ids, state_mask).values,
    'mean_nighttime_hours_above_21': compute_regional_mean(ds_night['hours_above_21'], region_6_ids, state_mask).values,
    'mean_nighttime_hours_above_24': compute_regional_mean(ds_night['hours_above_24'], region_6_ids, state_mask).values,
    'mean_vpd_mean': compute_regional_mean(ds_vpd['vpd_mean'], region_6_ids, state_mask).values,
    'mean_vpd_max': compute_regional_mean(ds_vpd['vpd_max'], region_6_ids, state_mask).values,
})

climate_data = pd.concat([r4_metrics, r6_metrics], ignore_index=True)

# Load cattle data
cattle_data = pd.read_csv(CATTLE_DATA_FILE, parse_dates=['date'])
cattle_data = cattle_data.rename(columns={'date': 'week_ending'})
cattle_data['region_4_total'] = cattle_data['region_4_beef_dairy']
cattle_data['region_6_total'] = cattle_data['region_6_beef_dairy']

cattle_r4 = cattle_data[['week_ending', 'region_4_total']].copy()
cattle_r4['region'] = 4
cattle_r4['total_beef_dairy'] = cattle_r4['region_4_total']
cattle_r4 = cattle_r4.drop(columns=['region_4_total'])

cattle_r6 = cattle_data[['week_ending', 'region_6_total']].copy()
cattle_r6['region'] = 6
cattle_r6['total_beef_dairy'] = cattle_r6['region_6_total']
cattle_r6 = cattle_r6.drop(columns=['region_6_total'])

cattle_regional = pd.concat([cattle_r4, cattle_r6], ignore_index=True)

# Merge
cattle_heat = pd.merge(cattle_regional, climate_data, on=['week_ending', 'region'], how='inner')
cattle_heat = cattle_heat.dropna()

# Add temporal features
cattle_heat['year'] = cattle_heat['week_ending'].dt.year
cattle_heat['month'] = cattle_heat['week_ending'].dt.month
cattle_heat['week_of_year'] = cattle_heat['week_ending'].dt.isocalendar().week

def get_season(month):
    if month in SEASONS['Winter']:
        return 'Winter'
    elif month in SEASONS['Spring']:
        return 'Spring'
    elif month in SEASONS['Summer']:
        return 'Summer'
    else:
        return 'Fall'

cattle_heat['season'] = cattle_heat['month'].apply(get_season)
cattle_focus = cattle_heat[cattle_heat['region'].isin([4, 6])].copy()
cattle_focus = cattle_focus.sort_values(['region', 'week_ending']).reset_index(drop=True)

print(f"Dataset loaded: {len(cattle_focus)} region-weeks")
print(f"Date range: {cattle_focus['year'].min()}-{cattle_focus['year'].max()}")

In [ ]:
# Recreate engineered features (same as modeling notebook)
heat_metrics = [
    'mean_daytime_hours_above_30',
    'mean_daytime_hours_above_35',
    'mean_nighttime_hours_above_21',
    'mean_nighttime_hours_above_24',
    'mean_vpd_mean',
    'mean_vpd_max'
]

# Create lagged features
for lag in range(1, 5):
    for metric in heat_metrics:
        lag_col = f'{metric}_lag{lag}'
        cattle_focus[lag_col] = cattle_focus.groupby('region')[metric].shift(lag)

# Create rolling averages
for window in [2, 4]:
    for metric in heat_metrics:
        roll_col = f'{metric}_roll{window}'
        cattle_focus[roll_col] = cattle_focus.groupby('region')[metric].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        )

# Create interaction features
cattle_focus['heat_vpd_interaction'] = (
    cattle_focus['mean_daytime_hours_above_30'] * cattle_focus['mean_vpd_mean']
)
cattle_focus['extreme_heat_poor_recovery'] = (
    cattle_focus['mean_daytime_hours_above_35'] * cattle_focus['mean_nighttime_hours_above_21']
)

# One-hot encode season
cattle_focus = pd.get_dummies(cattle_focus, columns=['season'], prefix='season', drop_first=False)

# Prepare model data
target_var = 'total_beef_dairy'
model_data = cattle_focus[all_features + [target_var, 'year']].dropna().copy()

# Split into train/test (same as before)
test_years = model_data['year'] > 2015
X_test = model_data.loc[test_years, all_features]
y_test = model_data.loc[test_years, target_var]

# Scale features
X_test_scaled = scaler.transform(X_test)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=all_features, index=X_test.index)

print(f"\nTest set prepared: {len(X_test)} samples")
print(f"Features: {len(all_features)}")

## 2. One-Dimensional Partial Dependence Plots

Examine the marginal effect of individual heat stress metrics on predicted cattle mortality.

In [ ]:
# Define key features for 1D PDP analysis
key_features = [
    'mean_daytime_hours_above_30',
    'mean_daytime_hours_above_35',
    'mean_nighttime_hours_above_21',
    'mean_nighttime_hours_above_24',
    'mean_vpd_mean',
    'mean_vpd_max'
]

# Get feature indices
feature_indices = [all_features.index(f) for f in key_features]

print("Computing 1D partial dependence...")
print(f"Model: {model_name}")
print(f"Features: {key_features}")

# Compute partial dependence
pd_results = partial_dependence(
    model,
    X_test_scaled,
    features=feature_indices,
    kind='average',
    grid_resolution=50
)

print("Partial dependence computed")

In [ ]:
# Create custom 1D PDP plots with enhanced visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

# Feature labels and units
feature_labels = {
    'mean_daytime_hours_above_30': 'Daytime Hours >30°C\n(hrs/week)',
    'mean_daytime_hours_above_35': 'Daytime Hours >35°C\n(hrs/week)',
    'mean_nighttime_hours_above_21': 'Nighttime Hours >21°C\n(hrs/week)',
    'mean_nighttime_hours_above_24': 'Nighttime Hours >24°C\n(hrs/week)',
    'mean_vpd_mean': 'Mean VPD\n(kPa)',
    'mean_vpd_max': 'Maximum VPD\n(kPa)'
}

# Critical thresholds from literature
critical_thresholds = {
    'mean_daytime_hours_above_30': [20, 30],  # Moderate to high stress
    'mean_daytime_hours_above_35': [5, 10],   # Severe stress onset
    'mean_nighttime_hours_above_21': [15, 25],  # Poor recovery
    'mean_nighttime_hours_above_24': [5, 15],   # Very poor recovery
    'mean_vpd_mean': [2.0, 2.5],               # High evaporative demand
    'mean_vpd_max': [3.0, 4.0]                 # Peak evaporative stress
}

for idx, (feat_idx, feature_name) in enumerate(zip(feature_indices, key_features)):
    ax = axes[idx]
    
    # Get PD values (already averaged across samples)
    pd_values = pd_results['average'][idx]
    grid_values = pd_results['grid_values'][idx]
    
    # Transform grid values back to original scale
    # Create a dummy array with all zeros except the feature of interest
    dummy_scaled = np.zeros((len(grid_values), len(all_features)))
    dummy_scaled[:, feat_idx] = grid_values
    dummy_original = scaler.inverse_transform(dummy_scaled)
    grid_original = dummy_original[:, feat_idx]
    
    # Plot partial dependence
    ax.plot(grid_original, pd_values, linewidth=3, color='darkblue', label='Partial Dependence')
    ax.fill_between(grid_original, pd_values.min(), pd_values, alpha=0.3, color='lightblue')
    
    # Add critical threshold lines
    if feature_name in critical_thresholds:
        thresholds = critical_thresholds[feature_name]
        ax.axvline(thresholds[0], color='orange', linestyle='--', linewidth=2, alpha=0.7,
                  label=f'Moderate Threshold ({thresholds[0]})')
        ax.axvline(thresholds[1], color='red', linestyle='--', linewidth=2, alpha=0.7,
                  label=f'High Threshold ({thresholds[1]})')
    
    # Formatting
    ax.set_xlabel(feature_labels[feature_name], fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted Mortality\n(1000 head)', fontsize=11, fontweight='bold')
    ax.set_title(f'Partial Dependence: {feature_name.replace("mean_", "").replace("_", " ").title()}',
                fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    
    # Add rug plot showing data distribution
    feature_data = X_test[feature_name].values
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    y_min, y_max = ax.get_ylim()
    rug_y = y_min + (y_max - y_min) * 0.02
    ax2.plot(feature_data, np.ones_like(feature_data) * rug_y, '|', 
            color='black', alpha=0.1, markersize=2)
    ax2.set_xticks([])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_partial_dependence_1d_key_features.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 01_partial_dependence_1d_key_features.png")

## 3. Individual Conditional Expectation (ICE) Plots

ICE plots show how predictions change for individual samples, revealing heterogeneity in feature effects.

In [ ]:
# Compute ICE curves for key features
print("Computing ICE (Individual Conditional Expectation) curves...")

# Sample subset for ICE (too many lines otherwise)
n_ice_samples = 100
ice_sample_indices = np.random.choice(len(X_test_scaled), size=n_ice_samples, replace=False)
X_ice = X_test_scaled[ice_sample_indices]

pd_ice_results = partial_dependence(
    model,
    X_ice,
    features=feature_indices,
    kind='individual',
    grid_resolution=50
)

print(f"ICE computed for {n_ice_samples} samples")

In [ ]:
# Plot ICE curves with average PD
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (feat_idx, feature_name) in enumerate(zip(feature_indices, key_features)):
    ax = axes[idx]
    
    # Get ICE values
    ice_values = pd_ice_results['individual'][idx]  # Shape: (n_samples, n_grid_points)
    grid_values = pd_ice_results['grid_values'][idx]
    
    # Transform grid to original scale
    dummy_scaled = np.zeros((len(grid_values), len(all_features)))
    dummy_scaled[:, feat_idx] = grid_values
    dummy_original = scaler.inverse_transform(dummy_scaled)
    grid_original = dummy_original[:, feat_idx]
    
    # Plot individual ICE curves (semi-transparent)
    for i in range(ice_values.shape[0]):
        ax.plot(grid_original, ice_values[i], color='gray', alpha=0.1, linewidth=0.5)
    
    # Plot average PD (thick line)
    pd_avg = ice_values.mean(axis=0)
    ax.plot(grid_original, pd_avg, linewidth=3, color='red', label='Average PD', zorder=10)
    
    # Add quantile bands (10th-90th percentile)
    pd_10 = np.percentile(ice_values, 10, axis=0)
    pd_90 = np.percentile(ice_values, 90, axis=0)
    ax.fill_between(grid_original, pd_10, pd_90, alpha=0.2, color='red',
                    label='10th-90th Percentile')
    
    # Formatting
    ax.set_xlabel(feature_labels[feature_name], fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted Mortality\n(1000 head)', fontsize=11, fontweight='bold')
    ax.set_title(f'ICE Plot: {feature_name.replace("mean_", "").replace("_", " ").title()}\n({n_ice_samples} samples)',
                fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_ice_plots_key_features.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 02_ice_plots_key_features.png")

## 4. Two-Dimensional Partial Dependence: Interaction Effects

2D PDPs reveal how pairs of features interact to affect mortality predictions.

In [ ]:
# Define key feature pairs for 2D PDP
interaction_pairs = [
    ('mean_daytime_hours_above_30', 'mean_vpd_mean'),           # Heat × VPD
    ('mean_daytime_hours_above_30', 'mean_nighttime_hours_above_21'),  # Heat × Recovery
    ('mean_vpd_mean', 'mean_nighttime_hours_above_21'),        # VPD × Recovery
    ('mean_daytime_hours_above_35', 'mean_vpd_max'),           # Extreme heat × Peak VPD
]

interaction_labels = [
    'Daytime Heat × VPD',
    'Daytime Heat × Nighttime Recovery',
    'VPD × Nighttime Recovery',
    'Extreme Heat × Peak VPD'
]

print("Computing 2D partial dependence for interaction effects...")
print(f"Feature pairs: {len(interaction_pairs)}")

In [ ]:
# Create 2D PDP plots
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, ((feat1, feat2), label) in enumerate(zip(interaction_pairs, interaction_labels)):
    ax = axes[idx]
    
    # Get feature indices
    feat1_idx = all_features.index(feat1)
    feat2_idx = all_features.index(feat2)
    
    # Compute 2D partial dependence
    pd_2d = partial_dependence(
        model,
        X_test_scaled,
        features=[(feat1_idx, feat2_idx)],
        kind='average',
        grid_resolution=30
    )
    
    # Get grid values and transform to original scale
    grid_values1 = pd_2d['grid_values'][0][0]
    grid_values2 = pd_2d['grid_values'][0][1]
    
    # Transform each feature separately
    dummy1 = np.zeros((len(grid_values1), len(all_features)))
    dummy1[:, feat1_idx] = grid_values1
    grid1_original = scaler.inverse_transform(dummy1)[:, feat1_idx]
    
    dummy2 = np.zeros((len(grid_values2), len(all_features)))
    dummy2[:, feat2_idx] = grid_values2
    grid2_original = scaler.inverse_transform(dummy2)[:, feat2_idx]
    
    # Get PD values
    pd_values_2d = pd_2d['average'][0].T  # Transpose for correct orientation
    
    # Create contour plot
    contour = ax.contourf(grid1_original, grid2_original, pd_values_2d, 
                          levels=20, cmap='RdYlGn_r', alpha=0.8)
    contour_lines = ax.contour(grid1_original, grid2_original, pd_values_2d,
                               levels=10, colors='black', alpha=0.3, linewidths=0.5)
    ax.clabel(contour_lines, inline=True, fontsize=8, fmt='%.0f')
    
    # Add colorbar
    cbar = plt.colorbar(contour, ax=ax)
    cbar.set_label('Predicted Mortality\n(1000 head)', fontsize=10, fontweight='bold')
    
    # Add scatter of actual data points (sample for clarity)
    n_scatter = 200
    scatter_indices = np.random.choice(len(X_test), size=min(n_scatter, len(X_test)), replace=False)
    ax.scatter(X_test[feat1].iloc[scatter_indices], 
              X_test[feat2].iloc[scatter_indices],
              s=5, color='black', alpha=0.2, edgecolors='none')
    
    # Labels
    ax.set_xlabel(feature_labels[feat1], fontsize=11, fontweight='bold')
    ax.set_ylabel(feature_labels[feat2], fontsize=11, fontweight='bold')
    ax.set_title(f'2D Partial Dependence: {label}', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3, color='white', linewidth=0.5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_partial_dependence_2d_interactions.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 03_partial_dependence_2d_interactions.png")

## 5. Threshold Detection: Statistical Identification of Critical Points

Use change-point detection to identify where mortality risk accelerates.

In [ ]:
# Detect critical thresholds using derivative analysis
print("Detecting critical thresholds using PD curve derivatives...")

# Recompute PD with higher resolution for better derivative estimation
pd_hires = partial_dependence(
    model,
    X_test_scaled,
    features=feature_indices,
    kind='average',
    grid_resolution=100
)

threshold_results = []

for idx, (feat_idx, feature_name) in enumerate(zip(feature_indices, key_features)):
    # Get PD values
    pd_values = pd_hires['average'][idx]
    grid_values = pd_hires['grid_values'][idx]
    
    # Transform to original scale
    dummy_scaled = np.zeros((len(grid_values), len(all_features)))
    dummy_scaled[:, feat_idx] = grid_values
    dummy_original = scaler.inverse_transform(dummy_scaled)
    grid_original = dummy_original[:, feat_idx]
    
    # Compute first derivative (rate of change)
    dy = np.diff(pd_values)
    dx = np.diff(grid_original)
    derivative = dy / dx
    
    # Compute second derivative (acceleration)
    d2y = np.diff(derivative)
    d2x = np.diff(grid_original[:-1])
    second_derivative = d2y / d2x
    
    # Find inflection points (where second derivative changes sign)
    sign_changes = np.where(np.diff(np.sign(second_derivative)))[0]
    
    # Find maximum acceleration point (steepest increase)
    max_accel_idx = np.argmax(second_derivative)
    threshold_accel = grid_original[1:-1][max_accel_idx]
    
    # Find elbow point (maximum derivative)
    max_deriv_idx = np.argmax(derivative)
    threshold_deriv = grid_original[:-1][max_deriv_idx]
    
    threshold_results.append({
        'Feature': feature_name,
        'Max_Acceleration_Threshold': threshold_accel,
        'Max_Derivative_Threshold': threshold_deriv,
        'N_Inflection_Points': len(sign_changes)
    })

threshold_df = pd.DataFrame(threshold_results)

print("\n" + "="*80)
print("DETECTED CRITICAL THRESHOLDS")
print("="*80)
print(threshold_df.to_string(index=False))

In [ ]:
# Visualize thresholds with derivatives
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (feat_idx, feature_name) in enumerate(zip(feature_indices, key_features)):
    ax = axes[idx]
    
    # Get PD values
    pd_values = pd_hires['average'][idx]
    grid_values = pd_hires['grid_values'][idx]
    
    # Transform to original scale
    dummy_scaled = np.zeros((len(grid_values), len(all_features)))
    dummy_scaled[:, feat_idx] = grid_values
    dummy_original = scaler.inverse_transform(dummy_scaled)
    grid_original = dummy_original[:, feat_idx]
    
    # Compute derivatives
    dy = np.diff(pd_values)
    dx = np.diff(grid_original)
    derivative = dy / dx
    
    # Create twin axes
    ax2 = ax.twinx()
    
    # Plot PD curve
    line1 = ax.plot(grid_original, pd_values, linewidth=3, color='darkblue', 
                    label='Partial Dependence')
    
    # Plot derivative
    line2 = ax2.plot(grid_original[:-1], derivative, linewidth=2, color='red', 
                     linestyle='--', alpha=0.7, label='Rate of Change (dPD/dx)')
    
    # Add threshold lines
    threshold_row = threshold_df[threshold_df['Feature'] == feature_name].iloc[0]
    ax.axvline(threshold_row['Max_Acceleration_Threshold'], color='orange', 
              linestyle=':', linewidth=2, alpha=0.7,
              label=f'Max Accel: {threshold_row["Max_Acceleration_Threshold"]:.1f}')
    ax.axvline(threshold_row['Max_Derivative_Threshold'], color='red', 
              linestyle=':', linewidth=2, alpha=0.7,
              label=f'Max Slope: {threshold_row["Max_Derivative_Threshold"]:.1f}')
    
    # Labels and formatting
    ax.set_xlabel(feature_labels[feature_name], fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted Mortality (1000 head)', fontsize=10, fontweight='bold', color='darkblue')
    ax2.set_ylabel('Rate of Change', fontsize=10, fontweight='bold', color='red')
    ax.set_title(f'Threshold Detection: {feature_name.replace("mean_", "").replace("_", " ").title()}',
                fontsize=12, fontweight='bold')
    ax.tick_params(axis='y', labelcolor='darkblue')
    ax2.tick_params(axis='y', labelcolor='red')
    
    # Combined legend
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    legend_lines, legend_labels = ax.get_legend_handles_labels()
    ax.legend(legend_lines + lines, legend_labels + labels, loc='upper left', fontsize=8)
    
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_threshold_detection_derivatives.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 04_threshold_detection_derivatives.png")

## 6. Bootstrap Confidence Intervals for Partial Dependence

Quantify uncertainty in PD estimates using bootstrap resampling.

In [ ]:
# Bootstrap confidence intervals
print("Computing bootstrap confidence intervals for partial dependence...")
print("(This may take a few minutes)")

n_bootstrap = 100  # Number of bootstrap samples
bootstrap_results = {feat: [] for feat in key_features}

for b in range(n_bootstrap):
    if (b + 1) % 20 == 0:
        print(f"  Bootstrap iteration {b+1}/{n_bootstrap}")
    
    # Resample test data with replacement
    boot_indices = np.random.choice(len(X_test_scaled), size=len(X_test_scaled), replace=True)
    X_boot = X_test_scaled[boot_indices]
    
    # Compute PD for this bootstrap sample
    pd_boot = partial_dependence(
        model,
        X_boot,
        features=feature_indices,
        kind='average',
        grid_resolution=50
    )
    
    # Store results
    for idx, feature_name in enumerate(key_features):
        bootstrap_results[feature_name].append(pd_boot['average'][idx])

print("Bootstrap complete")

In [ ]:
# Plot PD with confidence intervals
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (feat_idx, feature_name) in enumerate(zip(feature_indices, key_features)):
    ax = axes[idx]
    
    # Get original PD
    pd_values = pd_results['average'][idx]
    grid_values = pd_results['grid_values'][idx]
    
    # Transform to original scale
    dummy_scaled = np.zeros((len(grid_values), len(all_features)))
    dummy_scaled[:, feat_idx] = grid_values
    dummy_original = scaler.inverse_transform(dummy_scaled)
    grid_original = dummy_original[:, feat_idx]
    
    # Get bootstrap samples
    boot_samples = np.array(bootstrap_results[feature_name])
    
    # Compute confidence intervals
    ci_lower = np.percentile(boot_samples, 2.5, axis=0)
    ci_upper = np.percentile(boot_samples, 97.5, axis=0)
    ci_25 = np.percentile(boot_samples, 25, axis=0)
    ci_75 = np.percentile(boot_samples, 75, axis=0)
    
    # Plot confidence bands
    ax.fill_between(grid_original, ci_lower, ci_upper, alpha=0.2, color='blue',
                    label='95% CI')
    ax.fill_between(grid_original, ci_25, ci_75, alpha=0.3, color='blue',
                    label='IQR (25-75%)')
    
    # Plot mean PD
    ax.plot(grid_original, pd_values, linewidth=3, color='darkblue', 
           label='Partial Dependence')
    
    # Labels
    ax.set_xlabel(feature_labels[feature_name], fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted Mortality\n(1000 head)', fontsize=11, fontweight='bold')
    ax.set_title(f'PD with Confidence Intervals: {feature_name.replace("mean_", "").replace("_", " ").title()}',
                fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_partial_dependence_confidence_intervals.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 05_partial_dependence_confidence_intervals.png")

## 7. Summary and Biological Interpretation

In [ ]:
print("="*80)
print("KEY FINDINGS: PARTIAL DEPENDENCE ANALYSIS")
print("="*80)

print("\n1. CRITICAL THRESHOLDS DETECTED:")
print(threshold_df.to_string(index=False))

print("\n2. BIOLOGICAL INTERPRETATION:")
print("\n   A. DAYTIME HEAT (Hours >30°C):")
t30 = threshold_df[threshold_df['Feature'] == 'mean_daytime_hours_above_30']['Max_Derivative_Threshold'].values[0]
print(f"      - Critical threshold: {t30:.1f} hours/week")
print(f"      - Interpretation: Risk accelerates above ~{t30:.0f} hrs/week (≈{t30/7:.1f} hrs/day)")
print("      - Mechanism: Sustained heat load overwhelms thermoregulatory capacity")
print("      - Management: Provide shade/cooling before reaching this threshold")

print("\n   B. EXTREME DAYTIME HEAT (Hours >35°C):")
t35 = threshold_df[threshold_df['Feature'] == 'mean_daytime_hours_above_35']['Max_Derivative_Threshold'].values[0]
print(f"      - Critical threshold: {t35:.1f} hours/week")
print(f"      - Interpretation: Even brief extreme heat (>{t35:.0f} hrs/week) is dangerous")
print("      - Mechanism: Acute heat stroke risk, protein denaturation")
print("      - Management: Emergency protocols at any sustained exposure")

print("\n   C. POOR NIGHTTIME RECOVERY (Hours >21°C):")
t21 = threshold_df[threshold_df['Feature'] == 'mean_nighttime_hours_above_21']['Max_Derivative_Threshold'].values[0]
print(f"      - Critical threshold: {t21:.1f} hours/week")
print(f"      - Interpretation: Inadequate cooling time above ~{t21:.0f} hrs/week")
print("      - Mechanism: Cumulative heat storage, failed thermoregulatory recovery")
print("      - Management: Nighttime cooling (fans, sprinklers) is essential")

print("\n   D. VAPOR PRESSURE DEFICIT:")
tvpd = threshold_df[threshold_df['Feature'] == 'mean_vpd_mean']['Max_Derivative_Threshold'].values[0]
print(f"      - Critical threshold: {tvpd:.2f} kPa")
print(f"      - Interpretation: Evaporative cooling impaired above ~{tvpd:.1f} kPa")
print("      - Mechanism: High atmospheric water demand reduces respiratory/cutaneous cooling")
print("      - Management: Increase water access, use evaporative cooling systems")

print("\n3. INTERACTION EFFECTS (2D PDP Insights):")
print("\n   A. Heat × VPD Synergy:")
print("      - Combined effect exceeds additive expectations")
print("      - High VPD (>2.5 kPa) amplifies heat stress impacts")
print("      - Most dangerous: High heat (>25 hrs/wk at 30°C) + High VPD (>2.5 kPa)")

print("\n   B. Heat × Recovery Interaction:")
print("      - Poor nighttime recovery doubles daytime heat vulnerability")
print("      - Sequential hot days without cooling cause cumulative damage")
print("      - Critical combinations: >20 hrs daytime heat + >15 hrs poor recovery")

print("\n   C. VPD × Recovery Interaction:")
print("      - Warm nights (poor recovery) exacerbated by low humidity")
print("      - Nighttime VPD >1.5 kPa prevents effective heat dissipation")

print("\n4. NON-LINEAR RESPONSES:")
print("   - All features show accelerating risk curves (not linear)")
print("   - Thresholds represent points of rapid risk escalation")
print("   - Small increases above thresholds → large mortality increases")
print("   - Supports hypothesis H4: Asymmetric responses")

print("\n5. HETEROGENEITY (ICE Plot Insights):")
print("   - Wide variability in individual sample responses")
print("   - Some weeks highly sensitive, others resilient to same heat exposure")
print("   - Suggests importance of:")
print("     * Prior heat exposure (acclimation)")
print("     * Cattle breed/age composition")
print("     * Management practices (shade, water availability)")

print("\n" + "="*80)
print("MANAGEMENT RECOMMENDATIONS")
print("="*80)

print("\n1. MONITORING PRIORITIES:")
print(f"   - Alert when daytime heat exceeds {t30:.0f} hrs/week at >30°C")
print(f"   - Alert when nighttime recovery fails for >{t21:.0f} hrs/week")
print(f"   - Alert when VPD sustained above {tvpd:.1f} kPa")

print("\n2. INTERVENTION STRATEGIES:")
print("   - Install shade structures before summer (reduces daytime heat load)")
print("   - Implement nighttime cooling (fans, sprinklers) to ensure recovery")
print("   - Provide ad libitum cool water (10-15°C) during high VPD periods")
print("   - Reduce stocking density during compound heat + VPD + poor recovery events")

print("\n3. FORECASTING AND EARLY WARNING:")
print("   - Use weather forecasts to predict approaching threshold conditions")
print("   - 48-hour lead time sufficient for most interventions")
print("   - Focus on compound conditions (heat + VPD + poor recovery)")

print("\n4. BREED AND FACILITY ADAPTATION:")
print("   - Select heat-tolerant breeds for regions frequently exceeding thresholds")
print("   - Design facilities with cross-ventilation for nighttime cooling")
print("   - Consider shifting production season to avoid summer peaks")

print("\n" + "="*80)
print("")

# Export threshold results
threshold_df.to_csv(CATTLE_DATA_DIR / 'critical_thresholds_detected.csv', index=False)
print("Threshold results saved to: cattle_data/critical_thresholds_detected.csv")
print("\n✓ Analysis complete! All figures saved to figures/partial_dependence/")